In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from scipy.stats import skew
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

BACKEND_DIR = Path("..").resolve()
KB_DIR = BACKEND_DIR / "knowledge_base"

KB_JSON = KB_DIR / "knowledge_base.json"
UPITI_JSON = KB_DIR / "tekst_upiti.json"
UPITI_VAN_TEME_JSON = KB_DIR / "tekst_upiti_van_teme.json"
CENTRI_NPZ = KB_DIR / "bge_srednji_vektori.npz"

QDRANT_URL = "http://localhost:6333"
COLLECTION = "plant_text"

MODEL_NAME = "BAAI/bge-large-en-v1.5"
PREFIKS_UPITA = "Represent this sentence for searching relevant passages: "
BATCH_SIZE = 16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TEZINA_DOKUMENTA = 0.3

SEED = 42
B_BOOTSTRAP = 1000

L_RADNA = 3
CILJ_TACNOSTI = 95.0
MIN_DOBITAK_PP = 2.0

client = QdrantClient(url=QDRANT_URL, timeout=60)

In [2]:
srednji_vektor_val_upita = np.load(CENTRI_NPZ)["srednji_vektor_val_upita"]  

with open(KB_JSON, encoding="utf-8") as f:
    knowledge_base = json.load(f)

with open(UPITI_JSON, encoding="utf-8") as f:
    svi_upiti = json.load(f)

In [3]:
ime_vrste = {}
for sid in knowledge_base:
    ime_vrste[sid] = knowledge_base[sid]["scientific_name"]
BROJ_VRSTA = len(ime_vrste)

In [4]:
def ucitaj_upite(naziv):
    redovi = []
    for sid in svi_upiti:
        if sid == "_meta":
            continue
        for stavka in svi_upiti[sid][naziv]:
            redovi.append({
                "specie_id": sid,
                "scientific_name": svi_upiti[sid]["scientific_name"],
                "stil": stavka["stil"],
                "upit": stavka["upit"],
            })
    return pd.DataFrame(redovi)


val_df = ucitaj_upite("val")

print("Vrsta:", BROJ_VRSTA, "| val upita:", len(val_df))
val_df.head(4)

Vrsta: 324 | val upita: 648


,specie_id,scientific_name,stil,upit
0,0000,Lactuca virosa,short,Plant with purple black seeds and flowers bloo...
1,0000,Lactuca virosa,detailed,Tall plant growing up to almost 7 feet high wi...
2,0002,Pelargonium graveolens,short,Pale pink flowers on a multi-branched shrub wi...
3,0002,Pelargonium graveolens,detailed,Multi-branched plant with pale pink to almost ...


In [5]:
STOP_RECI = {
    "a", "about", "above", "after", "again", "all", "also", "am", "an", "and", "any", "are", "as",
    "at", "be", "because", "been", "before", "being", "below", "between", "both", "but", "by",
    "can", "could", "did", "do", "does", "doing", "down", "during", "each", "few", "for", "from",
    "further", "had", "has", "have", "having", "he", "her", "here", "hers", "him", "his", "how",
    "i", "if", "in", "into", "is", "it", "its", "itself", "just", "me", "more", "most", "my", "no",
    "nor", "not", "now", "of", "off", "on", "once", "only", "or", "other", "our", "out", "over",
    "own", "same", "she", "should", "so", "some", "such", "than", "that", "the", "their", "them",
    "then", "there", "these", "they", "this", "those", "through", "to", "too", "under", "until",
    "up", "very", "was", "we", "were", "what", "when", "where", "which", "while", "who", "whom",
    "why", "will", "with", "would", "you", "your", "around", "often", "usually", "like",
    "many", "much", "one", "two", "three", "cm", "mm", "m", "long", "wide",
}


def sadrzajne_reci(tekst):
    rezultat = set()
    for rec in re.findall(r"[a-z]+", tekst.lower()):
        if len(rec) >= 3 and rec not in STOP_RECI:
            rezultat.add(rec)
    return rezultat


def pokrivenost(reci_upita, reci_teksta):
    return len(reci_upita & reci_teksta) / len(reci_upita)


reci_kartice = {}
for sid in knowledge_base:
    reci_kartice[sid] = sadrzajne_reci(knowledge_base[sid]["tekst"])

svoja = []
tudja = []
for i, red in val_df.iterrows():
    reci_upita = sadrzajne_reci(red["upit"])

    svoja.append(pokrivenost(reci_upita, reci_kartice[red["specie_id"]]))
    
    ostale = []
    for sid in reci_kartice:
        if sid != red["specie_id"]:
            ostale.append(pokrivenost(reci_upita, reci_kartice[sid]))
    tudja.append(np.mean(ostale))

val_df["pokrivenost_svoja"] = svoja
val_df["pokrivenost_tudja"] = tudja
val_df["visak"] = val_df["pokrivenost_svoja"] - val_df["pokrivenost_tudja"]

kolone = ["pokrivenost_svoja", "pokrivenost_tudja", "visak"]
print(val_df[kolone].describe().round(3))
print()
print(val_df.groupby("stil")[kolone].mean().round(3))

       pokrivenost_svoja  pokrivenost_tudja    visak
count            648.000            648.000  648.000
mean               0.605              0.228    0.378
std                0.180              0.081    0.163
min                0.000              0.064   -0.378
25%                0.482              0.168    0.265
50%                0.593              0.205    0.376
75%                0.725              0.278    0.492
max                1.000              0.531    0.917

          pokrivenost_svoja  pokrivenost_tudja  visak
stil                                                 
detailed              0.527              0.175  0.352
short                 0.684              0.281  0.403


In [6]:
def centriraj(vektor, srednji_vektor):
    vektor = vektor - srednji_vektor
    return vektor / np.linalg.norm(vektor, axis=-1, keepdims=True)

In [7]:
model = SentenceTransformer(MODEL_NAME, device=DEVICE)

In [8]:
def skor_po_vrsti(vektor_upita, nivo):
    odgovor = client.query_points_groups(
        collection_name=COLLECTION,
        query=vektor_upita.tolist(),
        using="centriran",
        query_filter=Filter(must=[FieldCondition(key="nivo", match=MatchValue(value=nivo))]),
        group_by="specie_id",
        limit=BROJ_VRSTA,
        group_size=1,
        with_payload=False,
    )
    skorovi = {} 
    for grupa in odgovor.groups:
        skorovi[grupa.id] = grupa.hits[0].score
    return skorovi


def sirov_skor(sirovi_vektor_upita):
    odgovor = client.query_points(
        collection_name=COLLECTION,
        query=sirovi_vektor_upita.tolist(),
        using="sirov",
        query_filter=Filter(must=[FieldCondition(key="nivo", match=MatchValue(value="prozor"))]),
        limit=1,
        with_payload=False,
    )
    return odgovor.points[0].score


def pretrazi(upiti):
    sirovi_vektori_upita = model.encode([PREFIKS_UPITA + u for u in upiti],
                                  batch_size=BATCH_SIZE, normalize_embeddings=True)
    centrirani_vektori_upita = centriraj(sirovi_vektori_upita, srednji_vektor_val_upita)

    rangirane = []
    sirovi = []
    for i in tqdm(range(len(upiti))):
        prozor = skor_po_vrsti(centrirani_vektori_upita[i], "prozor")
        dokument = skor_po_vrsti(centrirani_vektori_upita[i], "dokument")

        vrste = []
        for sid in prozor:
            vrste.append((sid, (1 - TEZINA_DOKUMENTA) * prozor[sid] + TEZINA_DOKUMENTA * dokument[sid]))
        vrste.sort(key=lambda x: x[1], reverse=True)
        rangirane.append(vrste)

        sirovi.append(sirov_skor(sirovi_vektori_upita[i]))
    return rangirane, sirovi


def dodaj_rezultate(df, rangirane, sirovi):
    df = df.copy()
    df["sirov_score"] = sirovi
    for k in [1, 2, 3]:
        df[f"top{k}_specie_id"] = [vrste[k - 1][0] for vrste in rangirane]
        df[f"top{k}_score"] = [vrste[k - 1][1] for vrste in rangirane]
    df["top1_name"] = df["top1_specie_id"].map(ime_vrste)

    true_rank = []
    for sid, vrste in zip(df["specie_id"], rangirane):
        poredak = [v[0] for v in vrste]
        true_rank.append(poredak.index(sid) + 1)
    df["true_rank"] = true_rank

    df["gap"] = df["top1_score"] - df["top2_score"]
    df["top1_tacan"] = df["true_rank"] == 1
    df["tacna_u_listi"] = df["true_rank"] <= L_RADNA
    return df


val_rangirane_centrirani, val_sirovi = pretrazi(val_df["upit"].tolist())

val_df = dodaj_rezultate(val_df, val_rangirane_centrirani, val_sirovi)
val_df.to_csv(BACKEND_DIR / "rezultati/bge_large_rezultati.csv", index=False, encoding="utf-8-sig")

print(f"Top1 tačan (bez ikakvog praga): {val_df['top1_tacan'].mean():.1%}")
print(f"Tačna vrsta nije među prve {L_RADNA}: {(~val_df['tacna_u_listi']).mean():.1%}")
val_df[["scientific_name", "stil", "top1_name", "top1_score", "gap", "true_rank", "sirov_score"]].head(10).round(3)

100%|██████████| 648/648 [00:56<00:00, 11.54it/s]


Top1 tačan (bez ikakvog praga): 55.2%
Tačna vrsta nije među prve 3: 27.6%


,scientific_name,stil,top1_name,top1_score,gap,true_rank,sirov_score
0,Lactuca virosa,short,Hyoscyamus niger,0.323,0.000,4,0.697
1,Lactuca virosa,detailed,Epipactis atrorubens,0.398,0.065,3,0.826
2,Pelargonium graveolens,short,Pelargonium graveolens,0.253,0.004,1,0.691
3,Pelargonium graveolens,detailed,Pelargonium graveolens,0.371,0.096,1,0.753
4,Cirsium arvense,short,Trifolium stellatum,0.261,0.002,3,0.696
5,Cirsium arvense,detailed,Cirsium monspessulanum,0.212,0.015,2,0.749
6,Cirsium vulgare,short,Cirsium spinosissimum,0.281,0.015,12,0.704
7,Cirsium vulgare,detailed,Cirsium spinosissimum,0.383,0.105,2,0.796
8,Pelargonium peltatum,short,Atocion rupestre,0.254,0.030,6,0.693
9,Pelargonium peltatum,detailed,Pelargonium peltatum,0.199,0.001,1,0.760


In [9]:
proba = ("yellow flowers with five petals dotted with black glands, "
         "opposite sessile leaves with translucent dots")

rangirane, sirovi = pretrazi([proba])
for mesto, (sid, skor) in enumerate(rangirane[0][:5], start=1):
    print(mesto, ime_vrste[sid], round(skor, 3))
print("sirov skor:", round(sirovi[0], 3))

100%|██████████| 1/1 [00:00<00:00, 11.86it/s]

1 Hypericum humifusum 0.437
2 Hypericum maculatum 0.406
3 Hypericum pulchrum 0.392
4 Hypericum hirsutum 0.383
5 Hypericum androsaemum 0.312
sirov skor: 0.765


In [10]:
print(val_df[["sirov_score", "top1_score", "gap"]].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).round(4))

KANDIDATI_TAU = np.round(np.linspace(0.0, val_df["gap"].quantile(0.95), 31), 4)

print("Kandidati za TAU: ", KANDIDATI_TAU[0], "do", KANDIDATI_TAU[-1])

       sirov_score  top1_score       gap
count     648.0000    648.0000  648.0000
mean        0.7487      0.3699    0.0789
std         0.0473      0.0995    0.0843
min         0.6462      0.1844    0.0000
1%          0.6694      0.2196    0.0005
5%          0.6836      0.2463    0.0027
50%         0.7449      0.3489    0.0499
95%         0.8305      0.5572    0.2490
99%         0.8901      0.6801    0.4095
max         0.9395      0.7914    0.4753
Kandidati za TAU:  0.0 do 0.249


In [11]:
def hit_rate(rangovi, do_k=10):
    redovi = []
    for k in range(1, do_k + 1):
        redovi.append({
            "k": k,
            "hit_rate_%": (rangovi <= k).mean() * 100,
            "dobitak_pp": (rangovi == k).mean() * 100,
        })
    return pd.DataFrame(redovi).set_index("k")


hit_rate(val_df["true_rank"]).round(2)

,hit_rate_%,dobitak_pp
k,,
1,55.25,55.25
2,66.82,11.57
3,72.38,5.56
4,76.23,3.86
5,79.01,2.78
6,81.33,2.31
7,83.18,1.85
8,84.41,1.23
9,84.57,0.15


In [12]:
kratki = val_df[val_df["stil"] == "short"]
detaljni = val_df[val_df["stil"] == "detailed"]

po_stilu = pd.DataFrame({
    "short": hit_rate(kratki["true_rank"])["hit_rate_%"],
    "detailed": hit_rate(detaljni["true_rank"])["hit_rate_%"],
})
po_stilu.round(2)

,short,detailed
k,,
1,41.05,69.44
2,52.78,80.86
3,58.95,85.80
4,63.58,88.89
5,67.90,90.12
6,70.99,91.67
7,73.77,92.59
8,75.62,93.21
9,75.62,93.52


In [13]:
broj_prvih = val_df["top1_specie_id"].value_counts()
deset_posto = round(BROJ_VRSTA * 0.10)

print("Različitih vrsta na prvom mestu:", len(broj_prvih), "od", BROJ_VRSTA)
print(f"{deset_posto} najčešćih vrsta uzima {broj_prvih.head(deset_posto).sum() / len(val_df):.1%} prvih mesta")
print()

for k in [1, 5, 10]:
    N_k = {}
    for sid in ime_vrste:
        N_k[sid] = 0
    for vrste in val_rangirane_centrirani:
        for sid, skor in vrste[:k]:
            N_k[sid] += 1
    vrednosti = np.array(list(N_k.values()))
    print(f"k={k}: skewness N_k = {skew(vrednosti):.2f}, vrsta sa N_k = 0: {(vrednosti == 0).sum()}")

print("\nNajčešće prve vrste:")
print(broj_prvih.head(10).rename(index=ime_vrste))

Različitih vrsta na prvom mestu: 275 od 324
32 najčešćih vrsta uzima 29.5% prvih mesta

k=1: skewness N_k = 2.01, vrsta sa N_k = 0: 49
k=5: skewness N_k = 1.82, vrsta sa N_k = 0: 2
k=10: skewness N_k = 1.42, vrsta sa N_k = 0: 0

Najčešće prve vrste:
top1_specie_id
Maianthemum canadense         12
Butomus umbellatus            11
Kigelia africana              11
Erucastrum nasturtiifolium     9
Lavandula dentata              8
Osmunda regalis                7
Triadica sebifera              7
Ophrys speculum                7
Atocion rupestre               6
Pelargonium peltatum           6
Name: count, dtype: int64


In [14]:
opseg = pd.qcut(val_df["sirov_score"], 8)
tabela = val_df.groupby(opseg, observed=True)["tacna_u_listi"].agg(["size", "sum"])
tabela["nema_tacne_%"] = (1 - tabela["sum"] / tabela["size"]) * 100
tabela.round(1)

,size,sum,nema_tacne_%
sirov_score,,,
"(0.645, 0.696]",81,43,46.9
"(0.696, 0.715]",81,43,46.9
"(0.715, 0.729]",81,47,42.0
"(0.729, 0.745]",81,56,30.9
"(0.745, 0.76]",81,69,14.8
"(0.76, 0.777]",81,61,24.7
"(0.777, 0.8]",81,70,13.6
"(0.8, 0.94]",81,80,1.2


In [15]:
with open(UPITI_VAN_TEME_JSON, encoding="utf-8") as f:
    van_teme = json.load(f)["upiti"]

van_rangirane, van_sirovi = pretrazi(van_teme)
van_df = pd.DataFrame({
    "upit": van_teme,
    "top1_name": [ime_vrste[vrste[0][0]] for vrste in van_rangirane],
    "sirov_score": van_sirovi,
    "top1_score": [vrste[0][1] for vrste in van_rangirane],
})
van_df.to_csv(BACKEND_DIR / "rezultati/bge_large_van_teme.csv", index=False, encoding="utf-8-sig")


def oceni_prag(df, prag):
    odbijen = df["sirov_score"] < prag
    promasaj = ~df["tacna_u_listi"]
    if odbijen.sum() > 0:
        s_pravom = promasaj[odbijen].mean() * 100
    else:
        s_pravom = np.nan
    return {
        "PRAG": prag,
        "odbijenih_%": odbijen.mean() * 100,
        "odbijeno_s_pravom": (odbijen & promasaj).sum(),
        "odbijeno_bez_potrebe": (odbijen & ~promasaj).sum(),
        "s_pravom_medju_odbijenima_%": s_pravom,
        "uhvaceno_promasaja_%": (odbijen & promasaj).sum() / promasaj.sum() * 100,
    }


def izaberi_prag(df, van_teme_df):
    return round((van_teme_df["sirov_score"].max() + df["sirov_score"].min()) / 2, 4)


PRAG_ABSTAIN = izaberi_prag(val_df, van_df)

print(f"najveći sirov skor van teme: {van_df['sirov_score'].max():.4f}")
print(f"najmanji sirov skor val:     {val_df['sirov_score'].min():.4f}")
print(f"PRAG_ABSTAIN = {PRAG_ABSTAIN}\n")
print(f"odbijeno van teme: {(van_df['sirov_score'] < PRAG_ABSTAIN).mean():.1%}")
print(f"odbijeno val:      {(val_df['sirov_score'] < PRAG_ABSTAIN).mean():.1%}")

kandidati_poredjenje = np.round(np.linspace(val_df["sirov_score"].quantile(0.01),
                                            val_df["sirov_score"].quantile(0.99), 31), 4)
prag_df = pd.DataFrame([oceni_prag(val_df, p) for p in kandidati_poredjenje]).set_index("PRAG")
prag_df.round(1)

100%|██████████| 40/40 [00:03<00:00, 10.65it/s]

najveći sirov skor van teme: 0.6053
najmanji sirov skor val:     0.6462
PRAG_ABSTAIN = 0.6257

odbijeno van teme: 100.0%
odbijeno val:      0.0%


,odbijenih_%,odbijeno_s_pravom,odbijeno_bez_potrebe,s_pravom_medju_odbijenima_%,uhvaceno_promasaja_%
PRAG,,,,,
0.6694,1.1,3,4,42.9,1.7
0.6768,2.9,6,13,31.6,3.4
0.6841,5.6,15,21,41.7,8.4
0.6915,9.7,27,36,42.9,15.1
0.6988,15.3,44,55,44.4,24.6
0.7062,18.8,55,67,45.1,30.7
0.7136,24.1,72,84,46.2,40.2
0.7209,31.2,94,108,46.5,52.5
0.7283,37.2,109,132,45.2,60.9


In [16]:
upiti_prosli_prag= val_df[val_df["sirov_score"] >= PRAG_ABSTAIN]
print(f"Prošlo prvu kapiju: {len(upiti_prosli_prag)} upita ({len(upiti_prosli_prag) / len(val_df):.1%})\n")

opseg = pd.qcut(upiti_prosli_prag["gap"], 8)
tabela = upiti_prosli_prag.groupby(opseg, observed=True)["top1_tacan"].agg(["size", "sum"])
tabela["top1_tacan_%"] = tabela["sum"] / tabela["size"] * 100
tabela.round(1)

Prošlo prvu kapiju: 648 upita (100.0%)



,size,sum,top1_tacan_%
gap,,,
"(-0.0009558, 0.00602]",81,16,19.8
"(0.00602, 0.0163]",81,13,16.0
"(0.0163, 0.033]",81,29,35.8
"(0.033, 0.0499]",81,37,45.7
"(0.0499, 0.076]",81,45,55.6
"(0.076, 0.11]",81,64,79.0
"(0.11, 0.17]",81,76,93.8
"(0.17, 0.475]",81,78,96.3


In [20]:
def oceni_tau(upiti_prosli_prag, tau):
    siguran = upiti_prosli_prag["gap"] >= tau
    tacan = upiti_prosli_prag["top1_tacan"]
    
    return {
        "TAU": tau,
        "sigurnih_%": siguran.mean() * 100,
        "tacnost_sigurnih_%": tacan[siguran].mean() * 100,
        "sigurni_a_pogresni": (siguran & ~tacan).sum(),
        "nesigurni_a_tacni": (~siguran & tacan).sum(),
    }


def izaberi_tau(upiti_prosli_prag):
    for tau in KANDIDATI_TAU:
        if oceni_tau(upiti_prosli_prag, tau)["tacnost_sigurnih_%"] >= CILJ_TACNOSTI:
            return tau
    return np.inf


tau_df = pd.DataFrame([oceni_tau(upiti_prosli_prag, t) for t in KANDIDATI_TAU]).set_index("TAU")
rezultujuce_TAU = izaberi_tau(upiti_prosli_prag)

print("TAU =", rezultujuce_TAU)
tau_df.round(1)

TAU = 0.1162


,sigurnih_%,tacnost_sigurnih_%,sigurni_a_pogresni,nesigurni_a_tacni
TAU,,,,
0.0000,100.0,55.2,290,0
0.0083,84.6,62.0,208,18
0.0166,74.5,68.1,154,29
0.0249,68.1,71.4,126,43
0.0332,62.3,74.0,105,59
0.0415,56.0,78.0,80,75
0.0498,50.0,81.2,61,95
0.0581,44.9,85.2,43,110
0.0664,40.9,87.2,34,127


In [27]:
nesigurni = upiti_prosli_prag[upiti_prosli_prag["gap"] < rezultujuce_TAU]

hr_nesigurni = hit_rate(nesigurni["true_rank"], do_k=10)

DUZINA_LISTE = 3

print("Nesigurnih upita:", len(nesigurni))
print("DUZINA_LISTE =", DUZINA_LISTE)
print(f"tačna vrsta u listi od {DUZINA_LISTE} (nesigurni upiti): "
      f"{hr_nesigurni.loc[DUZINA_LISTE, 'hit_rate_%']:.1f}%")
hr_nesigurni.round(2)

Nesigurnih upita: 498
DUZINA_LISTE = 3
tačna vrsta u listi od 3 (nesigurni upiti): 65.1%


,hit_rate_%,dobitak_pp
k,,
1,43.17,43.17
2,58.03,14.86
3,65.06,7.03
4,69.88,4.82
5,73.29,3.41
6,76.10,2.81
7,78.51,2.41
8,80.12,1.61
9,80.32,0.20


In [29]:
def udeo(maska):
    return maska.mean() * 100 if len(maska) > 0 else np.nan


def simuliraj(df, prag, tau, duzina):
    odbijen = df["sirov_score"] < prag
    siguran = ~odbijen & (df["gap"] >= tau)
    lista = ~odbijen & ~siguran
    u_listi = df["true_rank"] <= duzina
    uspeh = (siguran & df["top1_tacan"]) | (lista & u_listi)
    return {
        "odbijeno_%": udeo(odbijen),
        "odbijeno_s_pravom_%": udeo(~u_listi[odbijen]),
        "siguran_%": udeo(siguran),
        "tacnost_sigurnih_%": udeo(df["top1_tacan"][siguran]),
        "lista_%": udeo(lista),
        "tacna_u_listi_%": udeo(u_listi[lista]),
        "uspeh_odgovorenih_%": udeo(uspeh[siguran | lista]),
        "uspeh_svih_upita_%": udeo(uspeh),
    }


sistem = simuliraj(val_df, PRAG_ABSTAIN, rezultujuce_TAU, DUZINA_LISTE)

print(f"PRAG_ABSTAIN = {PRAG_ABSTAIN} | TAU = {rezultujuce_TAU} | DUZINA_LISTE = {DUZINA_LISTE}\n")
print(pd.Series(sistem).round(2))
print(f"\nBez ikakvih pragova (uvek prva vrsta): {val_df['top1_tacan'].mean() * 100:.1f}%")

PRAG_ABSTAIN = 0.6257 | TAU = 0.1162 | DUZINA_LISTE = 3

odbijeno_%              0.00
odbijeno_s_pravom_%      NaN
siguran_%              23.15
tacnost_sigurnih_%     95.33
lista_%                76.85
tacna_u_listi_%        65.06
uspeh_odgovorenih_%    72.07
uspeh_svih_upita_%     72.07
dtype: float64

Bez ikakvih pragova (uvek prva vrsta): 55.2%


In [30]:
rng = np.random.default_rng(SEED)
indeksi_po_vrsti = val_df.groupby("specie_id").indices
vrste = list(indeksi_po_vrsti)

rezultati = []
for b in tqdm(range(B_BOOTSTRAP)):
    uzorak_vrsta = rng.choice(vrste, size=len(vrste), replace=True)
    indeksi = np.concatenate([indeksi_po_vrsti[v] for v in uzorak_vrsta])
    uzorak = val_df.iloc[indeksi].reset_index(drop=True)

    prag = izaberi_prag(uzorak, van_df)
    tau = izaberi_tau(uzorak[uzorak["sirov_score"] >= prag])

    mere = simuliraj(uzorak, prag, tau, DUZINA_LISTE)
    mere["PRAG_ABSTAIN"] = prag
    mere["TAU"] = tau
    rezultati.append(mere)

boot_df = pd.DataFrame(rezultati)
boot_df.to_csv(BACKEND_DIR / "eval_text_bootstrap.csv", index=False, encoding="utf-8-sig")

pd.DataFrame({
    "na svih 648": pd.Series({**sistem, "PRAG_ABSTAIN": PRAG_ABSTAIN, "TAU": rezultujuce_TAU}),
    "p2.5": boot_df.quantile(0.025),
    "p50": boot_df.quantile(0.5),
    "p97.5": boot_df.quantile(0.975),
}).round(3)

100%|██████████| 1000/1000 [00:15<00:00, 62.51it/s]


,na svih 648,p2.5,p50,p97.5
odbijeno_%,0.000,0.000,0.000,0.000
odbijeno_s_pravom_%,NaN,NaN,NaN,NaN
siguran_%,23.148,6.944,22.994,34.414
tacnost_sigurnih_%,95.333,95.000,95.740,100.000
lista_%,76.852,65.586,77.006,93.056
tacna_u_listi_%,65.060,59.012,64.922,71.736
uspeh_odgovorenih_%,72.068,68.364,72.068,75.617
uspeh_svih_upita_%,72.068,68.364,72.068,75.617
PRAG_ABSTAIN,0.626,0.626,0.626,0.634
TAU,0.116,0.091,0.116,0.216


In [ ]:
val_df["tercil"] = pd.qcut(val_df["visak"], 3, labels=["donji", "srednji", "gornji"])


def mere_podskupa(df):
    rezultat = {"upita": len(df)}
    rezultat.update(simuliraj(df, PRAG_ABSTAIN, rezultujuce_TAU, DUZINA_LISTE))
    rezultat["hit_rate@1_%"] = (df["true_rank"] <= 1).mean() * 100
    rezultat[f"hit_rate@{DUZINA_LISTE}_%"] = (df["true_rank"] <= DUZINA_LISTE).mean() * 100
    return pd.Series(rezultat)


print("Po stilu upita:")
print(pd.DataFrame({
    "short": mere_podskupa(val_df[val_df["stil"] == "short"]),
    "detailed": mere_podskupa(val_df[val_df["stil"] == "detailed"]),
}).round(1))

print("\nPo tercilu leksičkog preklapanja:")
print(pd.DataFrame({
    "donji": mere_podskupa(val_df[val_df["tercil"] == "donji"]),
    "srednji": mere_podskupa(val_df[val_df["tercil"] == "srednji"]),
    "gornji": mere_podskupa(val_df[val_df["tercil"] == "gornji"]),
}).round(1))

In [ ]:

oznake = [1] * len(val_df) + [0] * len(van_df)
skorovi_sirovi = list(val_df["sirov_score"]) + list(van_df["sirov_score"])
skorovi_centrirani = list(val_df["top1_score"]) + list(van_df["top1_score"])

auc_df = pd.DataFrame({
    "AUC": {
        "sirovi vektori": roc_auc_score(oznake, skorovi_sirovi),
        "centrirani vektori": roc_auc_score(oznake, skorovi_centrirani),
    },
    "val_min": {
        "sirovi vektori": val_df["sirov_score"].min(),
        "centrirani vektori": val_df["top1_score"].min(),
    },
    "van_teme_max": {
        "sirovi vektori": van_df["sirov_score"].max(),
        "centrirani vektori": van_df["top1_score"].max(),
    },
}).round(3)
auc_df.to_csv(BACKEND_DIR / "rezultati/bge_large_van_teme_auc.csv", index_label="skor", encoding="utf-8-sig")
print(auc_df)

van_max = van_df["sirov_score"].max()
val_min = val_df["sirov_score"].min()
KANDIDATI_PRAGA_VAN_TEME = np.round(np.linspace(van_max, val_df["sirov_score"].quantile(0.05), 21), 4)

odbijanje = pd.DataFrame({
    "PRAG": KANDIDATI_PRAGA_VAN_TEME,
    "odbijeno_van_teme_%": [(van_df["sirov_score"] < p).mean() * 100 for p in KANDIDATI_PRAGA_VAN_TEME],
    "odbijeno_val_%": [(val_df["sirov_score"] < p).mean() * 100 for p in KANDIDATI_PRAGA_VAN_TEME],
}).set_index("PRAG")
odbijanje.to_csv(BACKEND_DIR / "rezultati/bge_large_prag_van_teme.csv", encoding="utf-8-sig")

print(f"\nnajveći sirov skor van teme: {van_max:.4f} | najmanji sirov skor val: {val_min:.4f}")
print(f"izabrani PRAG_ABSTAIN (korak 6, sredina razmaka): {PRAG_ABSTAIN}\n")
odbijanje.round(1)

                      AUC  val_min  van_teme_max
sirovi vektori      1.000    0.646         0.605
centrirani vektori  0.853    0.184         0.368

najveći sirov skor van teme: 0.6053 | najmanji sirov skor val: 0.6462
izabrani PRAG_ABSTAIN (korak 6, sredina razmaka): 0.605



,odbijeno_van_teme_%,odbijeno_val_%
PRAG,,
0.6053,100.0,0.0
0.6092,100.0,0.0
0.6131,100.0,0.0
0.6170,100.0,0.0
0.6209,100.0,0.0
0.6249,100.0,0.0
0.6288,100.0,0.0
0.6327,100.0,0.0
0.6366,100.0,0.0
